<a href="https://colab.research.google.com/github/zimin205/25Analysis_BASE/blob/main/nDCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install pyarrow

In [7]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.3 MB/s eta 0:00:00


In [16]:
import unicodedata, os

def normalize_filenames_in_content():
    for fname in os.listdir("/content"):
        fixed = unicodedata.normalize("NFC", fname)
        if fixed != fname:
            os.rename(os.path.join("/content", fname),
                      os.path.join("/content", fixed))
            print(f"🔄 {fname} -> {fixed}")

normalize_filenames_in_content()


🔄 예시쿼리5.csv -> 예시쿼리5.csv


In [17]:
import pandas as pd
import numpy as np
import warnings
import os
from sklearn.metrics import ndcg_score

warnings.filterwarnings("ignore")

# ==== 하이퍼파라미터 / 설정 (예시쿼리 관련) ====
CONFIG = {
    "QUERY_FILES": [
        "예시쿼리1_final.csv",
        "예시쿼리2.csv",
        "예시쿼리3.csv",
        "예시쿼리4.csv",
        "예시쿼리5.csv",
    ],
    "QUERY_TEXTS": {
        0: "크기 1200에 컴퓨터랑 독서대를 같이 둘 수 있는, 화이트톤에 잘 어울리는 가성비 학생 책상을 추천해줘",
        1: "미드센츄리 무드의 바닥에 안 긁히고 공간을 차지하지 않는 내구성 좋은 아크릴 소재 의자를 추천해줘.",
        2: "4명이 앉을 수 있는 베이지 톤 소파 중에 내구성 좋고 얼룩 관리 쉬운 소파를 추천해줘",
        3: "원룸에서 사용할 충전단자와 조명이 있고, 수납과 조립이 편한 우드톤의 가성비 좋은 침대 프레임을 추천해줘",
        4: "전자레인지 밥솥이 올라갈 수 있는, 선이 안보이게 정리되고, 공간 절약형 깔끔한 수납장 추천해줘",
    },
    "MODELS": [
        ("koE5", "product", "desk_results - desk_results.csv"),
        ("koE5", "product", "chair_results - chair_results.csv"),
        ("koE5", "product", "sofa_results - sofa_results.csv"),
        ("koE5", "product", "bed_results - bed_results.csv"),
        ("koE5", "product", "cabinet_results - cabinet_results.csv"),
        ("koE5", "chunk", "desk_chunk - desk_chunk.csv"),
        ("koE5", "chunk", "chair_chunk - chair_chunk.csv"),
        ("koE5", "chunk", "sofa_chunk - sofa_chunk.csv"),
        ("koE5", "chunk", "bed_chunk - bed_chunk.csv"),
        ("koE5", "chunk", "cabinet_chunk - cabinet_chunk.csv"),
    ],
}
CONFIG["QUERY_FILE_BY_IDX"] = dict(enumerate(CONFIG["QUERY_FILES"]))

# ==== 데이터 로드/문서 만들기 ====
def load_data(data_csv="오늘의집_reviews.csv"):
    df = pd.read_csv(data_csv)
    cols = ["상품명", "옵션", "원가", "할인가"] + [f"리뷰{i}" for i in range(1, 11)]
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df

# ==== GT/키 생성 ====
OPTION_ALIASES = ["색상", "옵션명", "옵션"]

def _norm(x):
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except:
        pass
    return str(x).strip().lower()

def _get_option_value(row):
    for name in OPTION_ALIASES:
        if name in row and pd.notna(row[name]):
            v = _norm(row[name])
            if v:
                return v
    return ""

def build_key_from_row(row):
    product_name = _norm(row.get("상품명", ""))
    option = _get_option_value(row)
    return (product_name, option)

def load_gt_map_by_key(query_file):
    qdf = pd.read_csv(query_file)
    qdf.columns = [str(c).strip() for c in qdf.columns]

    score_col = None
    if "최종점수" in qdf.columns:
        score_col = "최종점수"
    elif "점수" in qdf.columns:
        score_col = "점수"
    else:
        qdf["_score_"] = 1.0
        score_col = "_score_"

    qdf[score_col] = pd.to_numeric(qdf[score_col], errors="coerce").fillna(0.0)
    qdf["_key_"] = qdf.apply(build_key_from_row, axis=1)

    gt_map = qdf.groupby("_key_")[score_col].max().to_dict()
    return gt_map

# ==== 키 매칭 개선 ====
def find_best_match_key(search_key, gt_map, threshold=0.7):
    search_product, search_option = search_key
    best_match = None
    best_score = 0.0
    best_similarity = 0.0

    for gt_key, gt_score in gt_map.items():
        gt_product, gt_option = gt_key

        if search_key == gt_key:
            return gt_key, gt_score, 1.0

        product_similarity = calculate_similarity(search_product, gt_product)
        option_similarity = calculate_similarity(search_option, gt_option)

        total_similarity = product_similarity * 0.7 + option_similarity * 0.3

        if total_similarity > best_similarity and total_similarity >= threshold:
            best_similarity = total_similarity
            best_match = gt_key
            best_score = gt_score

    return best_match, best_score, best_similarity

def calculate_similarity(str1, str2):
    if not str1 or not str2:
        return 0.0

    str1 = str1.lower().replace(' ', '')
    str2 = str2.lower().replace(' ', '')

    if str1 == str2:
        return 1.0

    if str1 in str2 or str2 in str1:
        return 0.8

    common_chars = set(str1) & set(str2)
    total_chars = set(str1) | set(str2)

    if not total_chars:
        return 0.0

    return len(common_chars) / len(total_chars)

# ==== 쿼리별 GT 파일 매핑 수정 ====
def get_correct_gt_file(query_idx, query_text):
    if query_text.startswith("크기 1200"):
        return CONFIG["QUERY_FILES"][0]
    elif query_text.startswith("미드센츄리"):
        return CONFIG["QUERY_FILES"][1]
    elif query_text.startswith("4명이 앉을 수"):
        return CONFIG["QUERY_FILES"][2]
    elif query_text.startswith("원룸에서 사용할"):
        return CONFIG["QUERY_FILES"][3]
    elif query_text.startswith("전자레인지"):
        return CONFIG["QUERY_FILES"][4]
    return None

# ==== Hybrid 모델 결과 로드 (경로 수정) ====
def load_hybrid_results(hybrid_file):
    try:
        if hybrid_file.endswith('.csv'):
            hybrid_results = pd.read_csv(hybrid_file)
            return hybrid_results
        else:
            print(f"지원하지 않는 파일 형식: {hybrid_file}")
            return None
    except Exception as e:
        print(f"Hybrid 결과 로드 실패: {e}")
        return None

# ==== output CSV 파일들에 쿼리 정보 추가 (기존 로직 삭제) ====
# 코랩 환경에서는 output 디렉토리가 없으므로 이 함수는 필요하지 않습니다.
# 대신, 직접 파일을 로드하고 평가하는 로직을 사용합니다.

# ==== 단순화된 Hybrid 모델 평가 ====
def evaluate_hybrid_model_simple(hybrid_results, df, df_doc_keys, model_name="hybrid"):
    rows = []
    key_to_index = {k: i for i, k in enumerate(df_doc_keys)}

    def select_prediction_score(row: pd.Series, model_name: str):
        search_type = None
        for name, st, csv_file in CONFIG["MODELS"]:
            if f"{name}_{st}" == model_name:
                search_type = st
                break

        if search_type is None:
            if '_product' in model_name:
                search_type = 'product'
            elif '_chunk' in model_name:
                search_type = 'chunk'

        if 'rrf_score' in row.index and pd.notna(row['rrf_score']):
            return float(row['rrf_score'])
        elif 'fused_score' in row.index and pd.notna(row['fused_score']):
            return float(row['fused_score'])
        else:
            return 0.0

    for idx, row in hybrid_results.iterrows():
        product_name = _norm(row.get("상품명", ""))
        option = _norm(row.get("옵션", ""))
        key = (product_name, option)
        doc_idx = row.get("product_row_id")

        if doc_idx is None or (isinstance(doc_idx, float) and pd.isna(doc_idx)):
            doc_idx = key_to_index.get(key)

        if doc_idx is None:
            continue

        score = select_prediction_score(row, model_name)

        rows.append({
            "model": model_name,
            "query_file": row.get("query_file", ""),
            "query_idx": row.get("query_idx", 0),
            "query_text": row.get("query_text", ""),
            "rank": row.get("rank", idx + 1),
            "doc_idx": int(doc_idx),
            "score": score,
            "key": key
        })
    return pd.DataFrame(rows)

# ==== NDCG 계산 ====
def compute_ndcg_for_group(group, df_doc_keys, gt_map, k=10):
    g = group.sort_values("rank").head(k)
    ranked_true = []
    ranked_pred = []

    for _, row in g.iterrows():
        doc_idx = int(row["doc_idx"])
        pred_s = float(row["score"])
        key = df_doc_keys[doc_idx]
        true_score = gt_map.get(key, 0.0)
        if true_score == 0.0:
            matched_key, matched_score, similarity = find_best_match_key(key, gt_map, threshold=0.6)
            if matched_key is not None and matched_score > 0.0:
                true_score = matched_score
        ranked_true.append(true_score)
        ranked_pred.append(pred_s)

    if not ranked_true:
        return 0.0

    if all(score == 0.0 for score in ranked_true):
        return 0.0

    if all(score == 0.0 for score in ranked_pred):
        return 0.0

    gt_coverage = sum(1 for score in ranked_true if score > 0.0) / len(ranked_true)
    gt_ranked = sorted(ranked_true, reverse=True)
    pred_ranked = sorted(ranked_pred, reverse=True)

    top3_gt = gt_ranked[:3]
    top3_pred = [ranked_pred[i] for i in range(min(3, len(ranked_pred)))]
    gt_top3_in_pred = sum(1 for gt_score in top3_gt if gt_score in top3_pred)
    order_accuracy = gt_top3_in_pred / min(3, len(top3_gt))

    y_true = np.array([ranked_true], dtype=float)
    y_score = np.array([ranked_pred], dtype=float)

    try:
        ndcg_result = ndcg_score(y_true, y_score, k=k)

        penalty_factor = 1.0
        if gt_coverage < 0.8:
            coverage_penalty = (0.8 - gt_coverage) * 0.3
            penalty_factor *= (1.0 - coverage_penalty)
        if order_accuracy < 0.66:
            order_penalty = (0.66 - order_accuracy) * 0.2
            penalty_factor *= (1.0 - order_penalty)

        final_ndcg = ndcg_result * penalty_factor
        return final_ndcg
    except Exception as e:
        return 0.0

# ==== 실제 준비 및 실행 ====
if __name__ == '__main__':
    df = load_data()
    df_doc_keys = [build_key_from_row(df.iloc[i]) for i in range(len(df))]
    print(f"✅ 문서 키 생성 완료: {len(df_doc_keys)}개")

    hybrid_results_list = []

    # 이미지에 보이는 파일명들을 사용하도록 수정
    for model_name, search_type, result_file in CONFIG["MODELS"]:
        print(f"\n[Hybrid] {model_name} ({search_type}) 결과 로드 중... ({result_file})")
        hybrid_results = load_hybrid_results(result_file)

        if hybrid_results is not None:
            hybrid_df = evaluate_hybrid_model_simple(
                hybrid_results, df, df_doc_keys, f"{model_name}_{search_type}"
            )

            if not hybrid_df.empty:
                hybrid_df['search_type'] = search_type
                hybrid_results_list.append(hybrid_df)
                print(f"  - {model_name} ({search_type}): {len(hybrid_df)}개 결과 로드")

    if hybrid_results_list:
        retrieval_df = pd.concat(hybrid_results_list, ignore_index=True)
        retrieval_path = "retrieval_results.parquet"
        retrieval_df.to_parquet(retrieval_path, index=False)
        print(f"\n✅ CSV 결과 저장 완료: {retrieval_path}  (rows={len(retrieval_df)})\n")
    else:
        print("⚠️ 로드된 CSV 결과가 없습니다.")
        retrieval_df = pd.DataFrame()

    results = []
    if not retrieval_df.empty:
        for qf, sub in retrieval_df.groupby("query_file"):
            print(f"--- 쿼리 파일: {qf} ---")
            if "query_text" in sub.columns:
                for (model, qidx), g in sub.groupby(["model", "query_idx"]):
                    query_text = g.iloc[0]["query_text"]
                    if not query_text or query_text.strip() == "":
                        continue

                    actual_gt_file = get_correct_gt_file(qidx, query_text)
                    if actual_gt_file is None:
                        continue

                    try:
                        gt_map = load_gt_map_by_key(actual_gt_file)
                        ndcg = compute_ndcg_for_group(g, df_doc_keys, gt_map, k=10)

                        model_parts = model.split('_')
                        if len(model_parts) >= 2:
                            base_model = '_'.join(model_parts[:-1])
                            search_type = model_parts[-1]
                        else:
                            base_model = model
                            search_type = "unknown"

                        results.append({
                            "query_file": actual_gt_file,
                            "base_model": base_model,
                            "search_type": search_type,
                            "model": model,
                            "query_idx": qidx,
                            "ndcg@10": ndcg
                        })
                    except Exception as e:
                        continue

        ndcg_df = pd.DataFrame(results)
        if not ndcg_df.empty:
            ndcg_df.to_csv("ndcg_per_query.csv", index=False)
            print("\n✅ NDCG 저장 완료: ndcg_per_query.csv (전체 상세 결과)")
        else:
            print("⚠️ NDCG 계산 결과가 없습니다.")

✅ 문서 키 생성 완료: 3309개

[Hybrid] koE5 (product) 결과 로드 중... (desk_results - desk_results.csv)
  - koE5 (product): 5개 결과 로드

[Hybrid] koE5 (product) 결과 로드 중... (chair_results - chair_results.csv)
  - koE5 (product): 5개 결과 로드

[Hybrid] koE5 (product) 결과 로드 중... (sofa_results - sofa_results.csv)
  - koE5 (product): 5개 결과 로드

[Hybrid] koE5 (product) 결과 로드 중... (bed_results - bed_results.csv)
  - koE5 (product): 5개 결과 로드

[Hybrid] koE5 (product) 결과 로드 중... (cabinet_results - cabinet_results.csv)
  - koE5 (product): 5개 결과 로드

[Hybrid] koE5 (chunk) 결과 로드 중... (desk_chunk - desk_chunk.csv)
  - koE5 (chunk): 5개 결과 로드

[Hybrid] koE5 (chunk) 결과 로드 중... (chair_chunk - chair_chunk.csv)
  - koE5 (chunk): 5개 결과 로드

[Hybrid] koE5 (chunk) 결과 로드 중... (sofa_chunk - sofa_chunk.csv)
  - koE5 (chunk): 5개 결과 로드

[Hybrid] koE5 (chunk) 결과 로드 중... (bed_chunk - bed_chunk.csv)
  - koE5 (chunk): 5개 결과 로드

[Hybrid] koE5 (chunk) 결과 로드 중... (cabinet_chunk - cabinet_chunk.csv)
  - koE5 (chunk): 5개 결과 로드

✅ CSV 결과 저장 완료: r